Imports and data loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay
)

In [2]:
df = pd.read_csv("classification_escalation.csv")
print(df.shape)
display(df.head())
print(df.info())
print(df["escalated"].value_counts(normalize=True).rename("proportion"))

(900, 6)


,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,9,7.7,low,web,21.6,0
1,2,6.1,high,web,21.8,0
2,1,2.5,medium,phone,28.8,0
3,7,5.3,low,email,52.5,0
4,4,7.1,medium,email,14.7,0


<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   complexity_score        900 non-null    int64  
 1   team_experience_years   869 non-null    float64
 2   priority                882 non-null    str    
 3   channel                 900 non-null    str    
 4   customer_tenure_months  878 non-null    float64
 5   escalated               900 non-null    int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 42.3 KB
None
escalated
0    0.793333
1    0.206667
Name: proportion, dtype: float64


count missing values

In [3]:
df.isna().sum()

complexity_score           0
team_experience_years     31
priority                  18
channel                    0
customer_tenure_months    22
escalated                  0
dtype: int64

### Step 1 – Findings

- **Shape:** The dataset has 900 rows (tickets) and 6 columns: 5 predictors and the target `escalated`.
- **Types:** `complexity_score` (integer), `team_experience_years` and `customer_tenure_months` (float) are numeric. `priority` (low/medium/high) and `channel` (email/web/phone) are categorical. `escalated` is a binary 0/1 integer target.
- **Missing values:** Missing values appear in `team_experience_years` (31), `priority` (18) and `customer_tenure_months` (22). The other columns are complete. These will be handled inside the preprocessing pipeline (median imputation for numeric, most-frequent for categorical).
- **Target prevalence:** 186 of 900 tickets (about 20.7%) were escalated and 714 (about 79.3%) were not. The classes are imbalanced, so accuracy alone will be misleading, because a model that never escalates would already score about 79%.

## Step 2 – Problem framing and train/test split

**Why classification?** The target `escalated` takes only two values (1 = escalated, 0 = not escalated), so we predict a discrete category rather than a continuous quantity. This makes it a binary classification problem.

**Positive class:** `escalated = 1`. Precision and recall are defined with respect to this class: recall is the share of truly escalating tickets we catch, and precision is the share of our escalation flags that are correct. We choose it as positive because missing an escalation (a false negative) is the costly error.

In [4]:
X = df.drop(columns="escalated")
y = df["escalated"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class counts:\n", y_train.value_counts())
print("Test class counts:\n", y_test.value_counts())
print("Train prevalence:", round(y_train.mean(), 4), "| Test prevalence:", round(y_test.mean(), 4))

Train: (675, 5) Test: (225, 5)
Train class counts:
 escalated
0    535
1    140
Name: count, dtype: int64
Test class counts:
 escalated
0    179
1     46
Name: count, dtype: int64
Train prevalence: 0.2074 | Test prevalence: 0.2044


**Why stratify?** The positive class is only about 21% of the data. Stratifying keeps that proportion nearly identical in train and test, so the test metrics are comparable and the test set contains enough positives (46) to evaluate recall and precision. The test set is held out and only used for final evaluation, not for fitting or imputing.

## Step 3 – Preprocessing

- **Numeric** (`complexity_score`, `team_experience_years`, `customer_tenure_months`): impute missing values with the **median** (robust to outliers), then **standardize** (zero mean, unit variance). Scaling matters for k-NN (distance-based) and helps logistic regression.
- **Categorical** (`priority`, `channel`): impute missing values with the **most frequent** category, then **one-hot encode**.
- Everything is wrapped in a `ColumnTransformer` that will sit inside each model's `Pipeline`. The imputers, scaler and encoder are therefore fit on training data only, which prevents leakage from the test set (and from the validation folds during cross-validation).

In [5]:
num_cols = ["complexity_score", "team_experience_years", "customer_tenure_months"]
cat_cols = ["priority", "channel"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

In [6]:
Xt = preprocess.fit_transform(X_train)
print("Transformed train shape:", Xt.shape)
print(list(preprocess.get_feature_names_out()))
print("Median imputation values:", preprocess.named_transformers_["num"]["imputer"].statistics_.round(2))
print("Most-frequent values:", preprocess.named_transformers_["cat"]["imputer"].statistics_)

Transformed train shape: (675, 9)
['num__complexity_score', 'num__team_experience_years', 'num__customer_tenure_months', 'cat__priority_high', 'cat__priority_low', 'cat__priority_medium', 'cat__channel_email', 'cat__channel_phone', 'cat__channel_web']
Median imputation values: [ 5.   4.4 19. ]
Most-frequent values: ['low' 'email']
